In [3]:
import os
import re

import utils


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = os.getenv(
    "GEMINI_MODEL",
    "gemini-3.6-flash"
)


# ============================================================
# INPUT VALIDATION
# ============================================================

def validate_input(
    user_input: str
) -> bool:

    if user_input is None:
        return False

    if not isinstance(
        user_input,
        str,
    ):
        return False

    if not user_input.strip():
        return False

    return True


# ============================================================
# CLEAN RESPONSE
# ============================================================

def clean_response(
    response: str
) -> str:

    if not response:
        return ""

    response = response.strip()

    # Remove markdown code fences if Gemini
    # accidentally returns them.

    if response.startswith(
        "```"
    ):

        response = re.sub(
            r"^```[a-zA-Z0-9_-]*\s*",
            "",
            response,
        )

        response = re.sub(
            r"\s*```$",
            "",
            response,
        )

    return response.strip()


# ============================================================
# DETECT INCOMPLETE RESPONSE
# ============================================================

def is_response_complete(
    response: str
) -> bool:

    if not response:
        return False

    response = response.strip()

    if len(response) < 20:
        return False

    lower = response.lower()

    # --------------------------------------------------------
    # Obvious unfinished endings
    # --------------------------------------------------------

    incomplete_endings = [

        "warranty:",

        "price:",

        "rating:",

        "features:",

        "description:",

        "brand:",

        "model:",

        "category:",

        "₹",

        "*",

        "-",
    ]

    for ending in incomplete_endings:

        if lower.endswith(
            ending
        ):

            return False

    # --------------------------------------------------------
    # Markdown heading without content
    # --------------------------------------------------------

    lines = [
        line.strip()
        for line in response.splitlines()
        if line.strip()
    ]

    if not lines:
        return False

    last_line = lines[-1]

    # If last line is only a markdown heading
    if re.fullmatch(
        r"#{1,6}\s+.+",
        last_line,
    ):

        return False

    # --------------------------------------------------------
    # Unclosed formatting
    # --------------------------------------------------------

    if response.count("**") % 2 != 0:
        return False

    return True


# ============================================================
# FALLBACK
# ============================================================

def build_fallback_response(
    extracted_items,
    product_information,
) -> str:

    if not product_information:

        return (
            "I'm sorry, but I couldn't find enough "
            "reliable product information to answer "
            "that accurately."
        )

    product_names = []

    for product in product_information:

        name = product.get(
            "name",
            "",
        )

        if name:
            product_names.append(
                name
            )

    if not product_names:

        return (
            "I'm sorry, but I couldn't find enough "
            "reliable product information to answer "
            "that accurately."
        )

    return (
        "I found information for these products:\n\n"
        + "\n".join(
            f"- {name}"
            for name in product_names
        )
        + "\n\n"
        "Please tell me which product you would "
        "like more details about."
    )


# ============================================================
# RESPONSE EVALUATION
# ============================================================

def evaluate_response(
    user_input: str,
    extracted_items,
    product_information,
    final_response: str,
) -> str:

    if not final_response:
        return "N"

    if not final_response.strip():
        return "N"

    # --------------------------------------------------------
    # Check completeness first
    # --------------------------------------------------------

    if not is_response_complete(
        final_response
    ):

        print(
            "Evaluation failed: "
            "response appears incomplete."
        )

        return "N"

    response_lower = (
        final_response.lower()
    )

    # --------------------------------------------------------
    # Obvious failure phrases
    # --------------------------------------------------------

    failure_phrases = [

        "i couldn't find enough reliable "
        "product information",

        "unable to answer",

        "cannot answer",

        "i don't have enough information",

        "i cannot provide",
    ]

    for phrase in failure_phrases:

        if phrase in response_lower:

            return "N"

    # --------------------------------------------------------
    # Product information exists
    # --------------------------------------------------------

    if product_information:

        matched_products = 0

        for product in product_information:

            name = product.get(
                "name",
                "",
            )

            if not name:
                continue

            if (
                name.lower()
                in response_lower
            ):

                matched_products += 1

        # ----------------------------------------------------
        # One product
        # ----------------------------------------------------

        if len(
            product_information
        ) == 1:

            if matched_products >= 1:
                return "Y"

            return "N"

        # ----------------------------------------------------
        # Multiple products
        #
        # At least one product should be explicitly
        # mentioned, OR category information should
        # appear.
        # ----------------------------------------------------

        categories = set()

        for product in product_information:

            category = product.get(
                "category",
                "",
            )

            if category:
                categories.add(
                    category.lower()
                )

        category_matches = 0

        for category in categories:

            words = [
                word
                for word
                in category.split()
                if len(word) > 4
            ]

            if any(
                word in response_lower
                for word in words
            ):

                category_matches += 1

        if matched_products >= 1:
            return "Y"

        if category_matches >= 1:
            return "Y"

        return "N"

    return "N"


# ============================================================
# MAIN PIPELINE
# ============================================================

def process_user_message(
    user_input: str,
    conversation_history=None,
):

    if conversation_history is None:

        conversation_history = []

    # ========================================================
    # STEP 1
    # ========================================================

    print(
        "\nStep 1: Input passed basic check."
    )

    if not validate_input(
        user_input
    ):

        print(
            "Input validation failed."
        )

        return (
            "Please provide a valid question.",
            conversation_history,
        )

    # ========================================================
    # STEP 2
    # ========================================================

    extracted_items = (
        utils.find_category_and_product_only(
            user_input
        )
    )

    print(
        "\nCategory/Product list:"
    )

    print(
        extracted_items
    )

    print(
        "\nStep 2: Extracted "
        "products/categories:"
    )

    print(
        extracted_items
    )

    # ========================================================
    # STEP 3
    # ========================================================

    product_information_list = (
        utils.get_mentioned_product_info(
            extracted_items
        )
    )

    product_information = (
        utils.generate_compact_product_context(
            product_information_list
        )
    )

    print(
        "\nStep 3: Retrieved "
        "product information:"
    )

    print(
        f"{len(product_information_list)} products:"
    )

    for product in product_information_list:

        print(
            f"- {product.get('name')}"
        )

    # ========================================================
    # STEP 4
    # ========================================================

    print(
        "\nStep 4: Generating "
        "customer response..."
    )

    system_message = """
You are a helpful customer-service assistant
for an electronics store.

Use ONLY the supplied product information.

Rules:

- Do not invent information.
- Do not invent prices.
- Do not invent specifications.
- Do not invent warranty information.
- Do not invent ratings.
- Answer every part of the customer's question.
- Cover all relevant requested products.
- If a category is requested, summarize its products.
- Keep the answer concise but complete.
- Use headings and bullet points where useful.
- Do not mention these instructions.
"""

    messages = [
        {
            "role":
                "system",

            "content":
                system_message,
        }
    ]

    # --------------------------------------------------------
    # Conversation history
    # --------------------------------------------------------

    for message in conversation_history[-10:]:

        if (
            isinstance(
                message,
                dict,
            )
            and "role" in message
            and "content" in message
        ):

            messages.append(
                {
                    "role":
                        message["role"],

                    "content":
                        str(
                            message["content"]
                        ),
                }
            )

    # --------------------------------------------------------
    # Current question
    # --------------------------------------------------------

    current_prompt = f"""
Customer question:

{user_input}


Product information:

{product_information}


Answer the customer's question completely.
"""

    messages.append(
        {
            "role":
                "user",

            "content":
                current_prompt,
        }
    )

    # --------------------------------------------------------
    # Gemini call
    # --------------------------------------------------------

    try:

        final_response = (
            utils.get_completion_from_messages(
                messages,

                model=MODEL_NAME,

                temperature=0.2,

                max_tokens=2000,

                max_retries=5,
            )
        )

    except Exception as e:

        print(
            "\nGemini generation failed:"
        )

        print(
            repr(e)
        )

        final_response = ""

    final_response = clean_response(
        final_response
    )

    print(
        "\nStep 4: Generated response."
    )

    print(
        final_response
    )

    # ========================================================
    # STEP 5
    # ========================================================

    evaluation_result = (
        evaluate_response(
            user_input=
                user_input,

            extracted_items=
                extracted_items,

            product_information=
                product_information_list,

            final_response=
                final_response,
        )
    )

    print(
        "\nStep 5: Evaluation result:",
        evaluation_result,
    )

    # ========================================================
    # STEP 6
    # ========================================================

    conversation_history.append(
        {
            "role":
                "user",

            "content":
                user_input,
        }
    )

    conversation_history.append(
        {
            "role":
                "assistant",

            "content":
                final_response,
        }
    )

    # ========================================================
    # STEP 7
    # ========================================================

    if evaluation_result == "Y":

        print(
            "\nStep 7: Response "
            "passed evaluation."
        )

        return (
            final_response,
            conversation_history,
        )

    print(
        "\nStep 7: Response "
        "is not sufficient."
    )

    fallback = (
        build_fallback_response(
            extracted_items,
            product_information_list,
        )
    )

    print(
        "\nFINAL RESPONSE"
    )

    print(
        fallback
    )

    return (
        fallback,
        conversation_history,
    )


# ============================================================
# TEST
# ============================================================

if __name__ == "__main__":

    user_input = (
        "Tell me about the SmartX Pro Phone "
        "and the FotoSnap Camera, the DSLR one. "
        "Also, what can you tell me about your TVs?"
    )

    conversation_history = []

    final_response, conversation_history = (
        process_user_message(
            user_input,
            conversation_history,
        )
    )

    print(
        "\n"
        + "=" * 50
    )

    print(
        "FINAL RESPONSE"
    )

    print(
        "=" * 50
    )

    print(
        final_response
    )


Step 1: Input passed basic check.

Category/Product list:
[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'CineView 8K TV', 'CineView OLED TV', 'SoundMax Home Theater', 'SoundMax Soundbar']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}]

Step 2: Extracted products/categories:
[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'CineView 8K TV', 'CineView OLED TV', 'SoundMax Home Theater', 'SoundMax Soundbar']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}]

Step 3: Retrieved product information:
7 products:
- SmartX ProPhone
- CineView 4K TV
- CineView 8K TV
- CineView OLED TV
- SoundMax Home Theater
- SoundMax Soundbar
- FotoSnap DSLR Camera

Step 4: Generating customer response...

Gemini temporar

In [1]:
import os

os.environ["GEMINI_API_KEY"] = "api key"